# 02 · Training BaseNet and ArmNet

Trains the two MLPs, saves them (with their normalisation stats) and checks them offline.
Closed-loop testing in the simulator is done with `sim_runner.py`, not here.

**How to use:** set the options in the *Configuration* cell, then **Restart & Run All**.

| Option | Meaning |
|---|---|
| `DATASET` | `"clean"` → expert demos only (`processed_step10.npz`), `"noise"` → + noise-injected demos (`processed_step10_noise.npz`) |
| `USE_DAGGER` | add the DAgger runs (`data/dagger_r*/`) to the **ArmNet** training set |
| `TRAIN_BASE`, `TRAIN_ARM` | which network(s) to (re)train |
| `VERSION` | name of this model version; checkpoints are saved as `<net>_<VERSION>.pt` **and** as the active `<net>.pt` used by `policy.py` / `sim_runner.py` |

Model versions so far (test set, both networks driving, see `results/`):

| Version | ArmNet training data | @5 cm | @2 cm | @1 cm |
|---|---|---|---|---|
| `clean` | expert demos | arm unstable (flies) | – | – |
| `noise` | + noise injection | arm holds, never starts reaching | – | – |
| `dagger1` | + DAgger round 1 | reaches (≈1.2 cm on ep 9030) | – | – |
| `dagger2_cut` | + DAgger round 2 (runs cut too early) | 95.9 % | 66.3 % | 12.2 % |
| `dagger2_full` | + DAgger round 2 (full runs) | 94.9 % | 83.7 % | 10.2 % |

## Configuration

In [ ]:
DATASET    = "noise"          # "clean" or "noise"
USE_DAGGER = True             # ArmNet only
TRAIN_BASE = False            # BaseNet already works -> only retrain when the base data changes
TRAIN_ARM  = True
VERSION    = "dagger2_full"   # name for the saved checkpoints

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys
import numpy as np
import torch
import matplotlib.pyplot as plt
torch.manual_seed(0); np.random.seed(0)
TASK_DIR = os.path.abspath("..")   # notebook runs from yousef/, so this is robotnik_arm_task/
RAW_DIR  = os.path.join(TASK_DIR, "data", "raw")
CKPT     = os.path.join(TASK_DIR, "checkpoints")
PROC     = os.path.join(TASK_DIR, "data", "processed_step10_noise.npz" if DATASET == "noise" else "processed_step10.npz")
sys.path.insert(0, os.path.join(TASK_DIR, "yousef"))

from models import MLP
from train import make_loaders, train_model
from dagger import add_dagger

device = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(CKPT, exist_ok=True)

D = dict(np.load(PROC))
print("dataset:", os.path.basename(PROC), "| base rows", D["Xb_train"].shape, "| arm rows", D["Xa_train"].shape, "|", device)
if USE_DAGGER:
    D = add_dagger(D)          # appends DAgger rows to Xa_train / Ya_train only

## Helpers: save, evaluate in physical units, learned stiffness

In [ ]:
def save_net(net, n, name):
    """n = "b" or "a". Saves weights + normalisation stats as <name>_<VERSION>.pt and as the active <name>.pt."""
    ck = {"state_dict": net.state_dict(),
          "in_dim": D[f"X{n}_mu"].shape[0], "out_dim": D[f"Y{n}_mu"].shape[0],
          "X_mu": D[f"X{n}_mu"], "X_sd": D[f"X{n}_sd"], "Y_mu": D[f"Y{n}_mu"], "Y_sd": D[f"Y{n}_sd"],
          "version": VERSION, "dataset": DATASET, "dagger": USE_DAGGER}
    torch.save(ck, os.path.join(CKPT, f"{name}_{VERSION}.pt"))
    torch.save(ck, os.path.join(CKPT, f"{name}.pt"))
    print(f"saved {name}_{VERSION}.pt and {name}.pt (active)")

def eval_physical(net, n, split="val"):
    """RMSE per output in physical units, overall and per expert phase (open loop, validation set)."""
    net.eval()
    x = (D[f"X{n}_{split}"] - D[f"X{n}_mu"]) / D[f"X{n}_sd"]
    with torch.no_grad():
        p = net(torch.from_numpy(x.astype(np.float32)).to(device)).cpu().numpy()
    err = (p * D[f"Y{n}_sd"] + D[f"Y{n}_mu"]) - D[f"Y{n}_{split}"]
    print("RMSE per output:", np.sqrt((err ** 2).mean(0)).round(4))
    for ph, name in enumerate(["align", "drive", "orient", "reaching"]):
        m = D[f"phase_{split}"] == ph
        print(f"  {name:9s} ({m.mean()*100:4.1f}% rows) RMSE:", np.sqrt((err[m] ** 2).mean(0)).round(4))

def plot_curves(hist, title):
    plt.figure(figsize=(5, 3.5))
    plt.plot(hist["train"], label="train"); plt.plot(hist["val"], label="val")
    plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("MSE (normalised)"); plt.title(title); plt.legend(); plt.show()

## 1. BaseNet — 10 inputs → (v, ω)
Skipped unless `TRAIN_BASE = True`.

In [ ]:
if TRAIN_BASE:
    train_dl, val_dl = make_loaders(D, "b")
    base_net, base_hist = train_model(MLP(10, 2), train_dl, val_dl, device=device)
    save_net(base_net, "b", "base_net")
    eval_physical(base_net, "b")          # v in m/s, omega in rad/s
    plot_curves(base_hist, f"BaseNet ({VERSION})")

## 2. ArmNet — 25 inputs → 7 torques

In [ ]:
if TRAIN_ARM:
    train_dl, val_dl = make_loaders(D, "a")
    arm_net, arm_hist = train_model(MLP(25, 7), train_dl, val_dl, device=device)
    save_net(arm_net, "a", "arm_net")
    eval_physical(arm_net, "a")           # torques in N·m
    plot_curves(arm_hist, f"ArmNet ({VERSION})")

## 3. Learned joint stiffness and damping (diagnostic)

The expert holds the arm with τ = Kp(q_init − q) − Kd·q̇ + g(q), **Kp = 100, Kd = 30**.
A stable learned controller must react the same way: nudging joint j by 0.01 rad should change its torque by ≈ −1 N·m.
Values near 0 mean the network does not correct deviations → the arm drifts/swings in closed loop.

Result for the `clean` ArmNet (3 Nov): joints 3 and 6 had **≈ 0 stiffness** (joint 6 also ≈ 0 damping) → unstable.

In [ ]:
from policy import BCPolicy

def stiffness_test(ep=9000, i=50):
    """Finite-difference dτ/dq and dτ/dq̇ of the ACTIVE checkpoints at a holding-phase state."""
    pol = BCPolicy(CKPT)
    e = np.load(os.path.join(RAW_DIR, f"ep_{ep:05d}.npz"))
    keys = ["base_pos", "base_bearing", "base_vel", "q_arm", "qd_arm", "ee_pos"]
    obs = {k: np.array(e[k][i], dtype=np.float64) for k in keys}; obs["target"] = e["target"]
    _, tau0 = pol(obs)
    for j in range(7):
        o = {k: v.copy() for k, v in obs.items()}; o["q_arm"][j] += 0.01
        _, tq = pol(o)
        o = {k: v.copy() for k, v in obs.items()}; o["qd_arm"][j] += 0.01
        _, tv = pol(o)
        print(f"joint {j+1}:  dτ/dq = {(tq[j]-tau0[j])/0.01:7.1f} (expert ≈ -100)   "
              f"dτ/dq̇ = {(tv[j]-tau0[j])/0.01:6.1f} (expert ≈ -30)")

stiffness_test()